# FMA split diagnostics

Set `VERSION` to the version printed by `atlas fma-split`. Set `COMPARE_VERSION` to another version built from the same inputs or to `None`. The tables compare the published FMA split, a seeded random assignment of groups and the group-stratified split. A pre-audit dataset uses the verified inventory and exact duplicates.

In [ ]:
import json

import pandas as pd
from IPython.display import display

from music_atlas.settings import paths

VERSION = "pre-audit-v1-8d68f7a1aa80ba0f"
COMPARE_VERSION = "pre-audit-rare-support-v1-9e3ce30b06c47c80"
data = paths()["data"]
ready = data / "ready"
result = ready / VERSION
if not (result / "tracks.parquet").is_file():
    available = sorted(p.parent.name for p in ready.glob("*/tracks.parquet"))
    raise ValueError(f"Set VERSION to one of: {', '.join(available)}")
summary = json.loads((result / "summary.json").read_text(encoding="utf-8"))
support = pd.read_csv(result / "class_support.csv")
versions = [VERSION]
if COMPARE_VERSION and COMPARE_VERSION != VERSION:
    comparison_dir = ready / COMPARE_VERSION
    if (comparison_dir / "tracks.parquet").is_file():
        other_summary = json.loads((comparison_dir / "summary.json").read_text(encoding="utf-8"))
        if all(summary[key] == other_summary[key] for key in ("inputs", "schema", "eligible")):
            versions.append(COMPARE_VERSION)
        else:
            print(f"{COMPARE_VERSION}: choose a comparison version with the same inputs.")
    else:
        print(f"{COMPARE_VERSION}: comparison manifest unavailable.")
status = "Audited dataset" if summary.get("audio_audited") else "Pre-audit dataset"
print(f"{VERSION} | {status}")
display(pd.DataFrame(summary["splits"]).T)
display(pd.DataFrame(summary["overlap"]).T)

## Label support and frequency baselines

The first table shows the largest shift in label prevalence in percentage points, relative to the selected annotated catalogue. The next tables count positive tracks and independent groups per class. The all-negative baseline can achieve high label accuracy by predicting the common absence of tags. Macro recall and macro-F1 give equal weight to each scored class and expose that failure.

In [ ]:
display(support.groupby(["scheme", "split"])["prevalence_shift"].apply(
    lambda values: 100 * values.abs().max()
).round(2).rename("Max absolute shift (pp)").unstack())
canonical = support[support["scheme"].eq("canonical")]
display(canonical.pivot(index="genre", columns="split", values=["tracks", "groups"]))
for split, scores in summary["baselines"].items():
    print(f"{split}: constant-score mAP {scores['constant_score_map']:.3f}")
    display(pd.DataFrame({k: v for k, v in scores.items() if isinstance(v, dict)}).T.round(3))

## Proportional and rare-support splits

Compare held-out positive tracks and independent groups per genre. Both versions partition the same catalogue. Use the three files from one version together.

In [ ]:
if len(versions) == 2:
    other_version = versions[1]
    other = pd.read_csv(ready / other_version / "class_support.csv")
    comparison = pd.concat([
        canonical.assign(version=VERSION),
        other[other["scheme"].eq("canonical")].assign(version=other_version),
    ])
    comparison = comparison[comparison["split"].isin(["validation", "test"])].copy()
    comparison["Tracks / groups"] = (
        comparison["tracks"].astype(str) + " / " + comparison["groups"].astype(str)
    )
    display(comparison.pivot(index="genre", columns=["version", "split"],
                             values="Tracks / groups"))

## Catalogue coverage and balanced evaluation

Coverage counts use the published manifest. Before the audio audit, duration is taken from FMA metadata and eligibility is provisional. Untagged tracks stay in the partition and are excluded from supervised genre evaluation.

The balanced diagnostic samples the same number of tagged and untagged-by-class tracks per scored genre in validation and test, with one track per independent group preferred. Both sides come from tracks with at least one annotation. One track can appear in several genre comparisons because FMA is multilabel. Use the full holdouts for primary metrics and the balanced view to examine per-class errors at equal support.

In [ ]:
manifests = {}
coverage = {}
balanced = {}
for version in versions:
    directory = ready / version
    columns = [
        "track_id", "split", "artist_id", "album_id",
        "duration_seconds", "genre_ids", "duplicate_group",
    ]
    tracks = pd.read_parquet(directory / "tracks.parquet", columns=columns)
    manifests[version] = tracks.loc[tracks["split"].notna()].set_index("track_id")
    coverage[version] = {
        "Catalogue rows": len(tracks),
        "Assigned tracks": len(manifests[version]),
        "Ineligible rows": tracks["split"].isna().sum(),
        "Missing artist": tracks["artist_id"].isna().sum(),
        "Missing album": tracks["album_id"].isna().sum(),
        "Missing duration": tracks["duration_seconds"].isna().sum(),
        "No genre annotation": tracks["genre_ids"].map(len).eq(0).sum(),
    }
    if (directory / "balanced_eval.parquet").is_file():
        view = pd.read_parquet(directory / "balanced_eval.parquet")
        for split, rows in view.groupby("split"):
            counts = rows.groupby(["genre_id", "label_present"]).size()
            assert counts.nunique() == 1
            balanced[version, split] = {
                "Scored genres": rows["genre_id"].nunique(),
                "Tracks per genre per side": counts.iloc[0],
                "Track–genre rows": len(rows),
            }
    else:
        print(f"{version}: balanced evaluation file unavailable.")
display(pd.DataFrame(coverage).T)
if balanced:
    display(pd.DataFrame(balanced).T.rename_axis(["Version", "Split"]))

## Follow-up to E1-04

E1-04 found that sparse genres can be dominated by a few artists and that duration cutoffs affect genres differently. Compare the same track IDs under the published assignment and the new splits. The label check compares the manifest's ancestor expansion with FMA's published `genres_all`, which E1-04 used for its inclusive counts. [E1-04 metadata exploration](https://github.com/piotrczech/music-atlas/blob/master/notebooks/fma_metadata.ipynb)

The grouped allocator keeps artists and duplicate groups within a split. A release with multiple artists can still span splits, so the album count measures a separate source of dependence.

The last tables follow the five least common top-level genres and the three most affected by a 10-minute cutoff in the selected catalogue. Read the largest artist's share alongside the number of tracks and artists to see whether extra tracks diversify the evaluation. The duration columns reproduce the E1-04 cutoffs using published durations and show what would be lost in each partition.

In [ ]:
from ast import literal_eval

if manifests:
    metadata = data / "fma/fma_metadata"
    fields = {
        ("track", "genres_all"): "published_genres",
        ("track", "duration"): "published_duration",
        ("set", "split"): "published_split",
    }
    original = pd.read_csv(
        metadata / "tracks.csv", index_col=0, header=[0, 1], low_memory=False
    )[list(fields)].copy()
    original.columns = list(fields.values())
    original["published_genres"] = (
        original["published_genres"].fillna("[]").map(literal_eval).map(frozenset)
    )
    original["published_split"] = original["published_split"].replace("training", "train")
    genres = pd.read_csv(metadata / "genres.csv", index_col="genre_id")
    roots = genres.loc[genres["parent"].eq(0), "title"]
    root_ids = frozenset(roots.index)

    agreement = {}
    leakage = {}
    branch_profiles = {}
    for version, assigned in manifests.items():
        rows = assigned.join(original, validate="one_to_one")
        reference_tags = rows["published_genres"].map(
            lambda tags: tags if isinstance(tags, frozenset) else frozenset()
        )
        manifest_tags = rows["genre_ids"].map(frozenset)
        agreement[version] = {
            "Tracks compared": len(rows),
            "Missing FMA metadata rows": len(rows.index.difference(original.index)),
            "Different expanded tags": manifest_tags.ne(reference_tags).sum(),
            "Different top-level tags": (
                manifest_tags.map(root_ids.intersection)
                .ne(reference_tags.map(root_ids.intersection)).sum()
            ),
        }

        assignments = {"Published FMA": "published_split", "Versioned": "split"}
        for assignment, split_column in assignments.items():
            leakage[version, assignment] = {
                label: rows.groupby(field)[split_column].nunique().gt(1).sum()
                for field, label in {
                    "artist_id": "Artists", "album_id": "Albums",
                    "duplicate_group": "Duplicate groups",
                }.items()
            }

        duration = rows["published_duration"]
        branches = rows.assign(
            genre_id=reference_tags,
            short=duration.gt(0) & duration.lt(10),
            long=duration.gt(600) & duration.lt(float("inf")),
        ).explode("genre_id")
        branch_profiles[version] = branches.loc[
            branches["genre_id"].isin(roots.index),
            ["split", "genre_id", "artist_id", "short", "long"],
        ]

    display(pd.DataFrame(agreement).T)
    display(pd.DataFrame(leakage).T.rename_axis(["Version", "Assignment"]))

In [ ]:
def branch_profile(rows):
    keys = ["split", "genre_id"]
    stats = rows.groupby(keys).agg(
        tracks=("artist_id", "size"),
        artists=("artist_id", "nunique"),
        short=("short", "mean"),
        long=("long", "mean"),
    )
    top_artist = rows.groupby([*keys, "artist_id"]).size().groupby(level=[0, 1]).max()
    stats["top_artist"] = top_artist / stats["tracks"]
    return stats


if manifests:
    for version, branches in branch_profiles.items():
        rows = pd.concat([branches.assign(split="Catalogue"), branches])
        stats = branch_profile(rows)
        catalogue = stats.loc["Catalogue"]
        focus = catalogue.nsmallest(5, "tracks").index.union(
            catalogue.nlargest(3, "long").index, sort=False
        )
        profile = stats.loc[(slice(None), focus), :].reset_index()
        profile["Genre"] = profile["genre_id"].map(roots)
        profile[["short", "long", "top_artist"]] *= 100
        profile = profile.rename(columns={
            "short": "Under 10 s (%)", "long": "Over 10 min (%)",
            "top_artist": "Top artist (%)", "tracks": "Tracks", "artists": "Artists",
        })
        print(version)
        display(profile.drop(columns="genre_id").set_index(["Genre", "split"])
                .reindex(pd.MultiIndex.from_product(
                    [roots.loc[focus], ["Catalogue", "train", "validation", "test"]],
                    names=["Genre", "split"],
                )).round(1))